In [7]:
import sys
import os
from pprint import pprint

sys.path.append(os.path.dirname(os.getcwd()))
# from src.core import * # type: ignore
from src.hamiltonians import *
from src.lattices import *
from src.propagators import *
from src.states import *
from src.walkers import *

In [8]:
Nsites = 2
Lx=10
Ly=10
Nup = Nsites // 2
Ndown = Nsites // 2
# Nup = Lx * Ly // 2
# Ndown = Lx * Ly // 2
# Nup = Lx * Ly // (2 * 8) 
# Ndown = Lx * Ly // (2 * 8)
t = 1.0
U = 0.0
dtau = 0.05
n_steps = 10000

In [9]:
lattice = Chain(
    n_sites=Nsites,
    pbc=False
)
# lattice = Square(
#     Lx=Lx,
#     Ly=Lx,
#     pbc=True
# )

system = HubbardSystem(
    lattice=lattice,
    t=t,
    U=U
)

K = system.h_kin
# K

In [10]:
trial = SlaterDeterminantTwoSpinState(
    hamiltonian=system,
    n_electrons_up=Nup,
    n_electrons_down=Ndown
)

trial.initialize("non-interacting")

evals = system.get_non_interacting_evals_evecs(
    return_evals=True,
    return_evecs=False
)

E_exact = (
    np.sum(evals[:Nup])
    +
    np.sum(evals[:Ndown])
)

print()
print("Exact U=0 ground-state energy")
print(E_exact)
print(evals)


Exact U=0 ground-state energy
-2.0
[-1.  1.]


In [11]:
state = SlaterDeterminantTwoSpinState(
    hamiltonian=system,
    n_electrons_up=Nup,
    n_electrons_down=Ndown
)

state.initialize("random")

walker = Walker(state)

prop = HubbardPropagator(
    K=K,
    U=U,
    dtau=dtau
)

print()
print("gamma =", prop.gamma)
print()

for step in range(n_steps):

    prop.propagate(walker)

    walker.orthogonalize()

    if step % 20 == 0:

        overlap = trial.overlap_calculation_logdet(
            walker.state
        )

        print(
            f"step = {step:4d}"
            f"   overlap = {abs(overlap):.12f}"
        )


final_overlap = trial.overlap_calculation_logdet(
    walker.state
)

print()
print("Final overlap:")
print(abs(final_overlap))

error_up = np.linalg.norm(
    walker.state.phi_up @ walker.state.phi_up.conj().T
    -
    trial.phi_up @ trial.phi_up.conj().T
)

error_down = np.linalg.norm(
    walker.state.phi_down @ walker.state.phi_down.conj().T
    -
    trial.phi_down @ trial.phi_down.conj().T
)

print()
print("Projector error (up)   =", error_up)
print("Projector error (down) =", error_down)



gamma = 0.0

step =    0   overlap = 0.812256069239
step =   20   overlap = 0.995562446014
step =   40   overlap = 0.999918274853
step =   60   overlap = 0.999998503000
step =   80   overlap = 0.999999972581
step =  100   overlap = 0.999999999498
step =  120   overlap = 0.999999999991
step =  140   overlap = 1.000000000000
step =  160   overlap = 1.000000000000
step =  180   overlap = 1.000000000000
step =  200   overlap = 1.000000000000
step =  220   overlap = 1.000000000000
step =  240   overlap = 1.000000000000
step =  260   overlap = 1.000000000000
step =  280   overlap = 1.000000000000
step =  300   overlap = 1.000000000000
step =  320   overlap = 1.000000000000
step =  340   overlap = 1.000000000000
step =  360   overlap = 1.000000000000
step =  380   overlap = 1.000000000000
step =  400   overlap = 1.000000000000
step =  420   overlap = 1.000000000000
step =  440   overlap = 1.000000000000
step =  460   overlap = 1.000000000000
step =  480   overlap = 1.000000000000
step =  500

In [12]:
def energy(phi, K):
    return np.trace(phi.conj().T @ K @ phi).real

E_proj = (energy(walker.state.phi_up, K) + energy(walker.state.phi_down, K))
E_proj = system.calculate_energy(walker)

print(E_proj)
print(E_exact)
print(np.abs((E_proj - E_proj) / E_exact) * 100)

-2.0
-2.0
0.0
